# Gold Fact: Weather

Build the historical London weather fact table.

This notebook:

1. Reads validated Silver weather observations.
2. Resolves each observation to the London-local calendar date.
3. Preserves the deterministic Silver observation key.
4. Applies Gold data-quality checks.
5. Writes observations using an idempotent insert-only Delta merge.

**Source:** `workspace.urbanpulse_silver.weather`

**Target:** `workspace.urbanpulse_gold.fact_weather`

**Grain:** One weather observation per location and observation timestamp.

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import dependencies

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.fact_weather import (
    build_fact_weather,
)

from urbanpulse.quality.fact_weather import (
    invalid_fact_weather,
)

from urbanpulse.utils.delta import (
    merge_insert_only,
)

## 3. Define source and target tables

In [0]:
SILVER_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "weather"
)

DIM_DATE = (
    "workspace."
    "urbanpulse_gold."
    "dim_date"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "fact_weather"
)

## 4. Read Silver weather observations and date dimension

In [0]:
silver_df = spark.table(
    SILVER_TABLE
)

dim_date_df = spark.table(
    DIM_DATE
)

source_count = silver_df.count()

print(
    f"Silver weather observations: "
    f"{source_count}"
)

## 5. Validate Silver observation grain

`weather_observation_key` must uniquely identify each weather observation.

In [0]:
duplicate_source_keys_df = (
    silver_df
    .groupBy(
        "weather_observation_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_source_count = (
    duplicate_source_keys_df.count()
)

if duplicate_source_count > 0:
    display(
        duplicate_source_keys_df
    )

    raise ValueError(
        "Duplicate Silver weather "
        "observation keys detected."
    )

print(
    "Silver weather grain validation passed."
)

## 6. Resolve observation dates

Weather timestamps are stored in UTC.

The date dimension relationship is based on the corresponding Europe/London calendar date.

In [0]:
fact_df = build_fact_weather(
    silver_df=silver_df,
    dim_date_df=dim_date_df,
)

fact_count = fact_df.count()

print(
    f"Silver rows: {source_count}"
)

print(
    f"Fact rows:   {fact_count}"
)

display(
    fact_df
    .orderBy(
        F.col(
            "weather_observed_at"
        ).desc()
    )
)

## 7. Validate date dimension resolution

Every Silver weather observation must resolve to exactly one date dimension row.

In [0]:
if fact_count != source_count:
    raise ValueError(
        "Fact row count does not match "
        "Silver weather row count. "
        "Investigate date dimension resolution."
    )

print(
    "All weather observations resolved "
    "exactly once."
)

## 8. Apply fact quality checks

In [0]:
invalid_df = (
    invalid_fact_weather(
        fact_df
    )
)

invalid_count = (
    invalid_df.count()
)

print(
    f"Invalid facts: "
    f"{invalid_count}"
)

if invalid_count > 0:
    display(
        invalid_df
    )

    raise ValueError(
        f"{invalid_count} invalid "
        "weather facts detected."
    )

print(
    "Weather fact quality checks passed."
)

## 9. Validate fact keys

In [0]:
duplicate_fact_keys_df = (
    fact_df
    .groupBy(
        "weather_observation_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_fact_keys = (
    duplicate_fact_keys_df.count()
)

if duplicate_fact_keys > 0:
    display(
        duplicate_fact_keys_df
    )

    raise ValueError(
        "Duplicate weather fact "
        "keys detected."
    )

print(
    "Weather fact keys are unique."
)

## 10. Add Gold processing metadata

In [0]:
gold_df = (
    fact_df
    .withColumn(
        "created_at",
        F.current_timestamp(),
    )
)

## 11. Merge weather observations into Gold

Weather observations are insert-only.

Previously processed observations are not rewritten during reruns.

In [0]:
merge_insert_only(
    spark=spark,
    source_df=gold_df,
    target_table=TARGET_TABLE,
    merge_condition="""
        target.weather_observation_key
        =
        source.weather_observation_key
    """,
)

print(
    f"Gold fact updated: "
    f"{TARGET_TABLE}"
)

## 12. Verify weather facts

In [0]:
%sql
SELECT
    weather_observation_key,
    observation_date_key,
    weather_observed_at,
    weather_observed_at_local,
    temperature_c,
    relative_humidity_pct,
    precipitation_mm,
    weather_description,
    cloud_cover_pct,
    wind_speed_kmh,
    wind_gusts_kmh
FROM workspace.urbanpulse_gold.fact_weather
ORDER BY weather_observed_at DESC;

In [0]:
%sql
-- Verify Date relationships
SELECT
    d.calendar_date,
    d.day_name,
    d.is_weekend,
    d.is_bank_holiday,
    w.weather_observed_at,
    w.temperature_c,
    w.precipitation_mm,
    w.weather_description
FROM workspace.urbanpulse_gold.fact_weather w

INNER JOIN workspace.urbanpulse_gold.dim_date d
    ON w.observation_date_key = d.date_key

ORDER BY
    w.weather_observed_at DESC;

In [0]:
%sql
-- Referential integrity check
SELECT w.*
FROM workspace.urbanpulse_gold.fact_weather w

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_date d
    ON w.observation_date_key = d.date_key;

In [0]:
%sql
-- Verify London-local date assignment
SELECT
    weather_observed_at,
    weather_observed_at_local,
    observation_date_key,
    DATE(
        FROM_UTC_TIMESTAMP(
            weather_observed_at,
            'Europe/London'
        )
    ) AS expected_local_date,
    d.calendar_date

FROM workspace.urbanpulse_gold.fact_weather w

INNER JOIN workspace.urbanpulse_gold.dim_date d
    ON w.observation_date_key = d.date_key

WHERE
    DATE(
        FROM_UTC_TIMESTAMP(
            weather_observed_at,
            'Europe/London'
        )
    ) <> d.calendar_date;

In [0]:
%sql
-- Validate observation-key uniqueness
SELECT
    weather_observation_key,
    COUNT(*) AS records
FROM workspace.urbanpulse_gold.fact_weather
GROUP BY weather_observation_key
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Inspecting weather history
SELECT
    d.calendar_date,

    COUNT(*) AS observations,

    ROUND(
        AVG(w.temperature_c),
        2
    ) AS avg_temperature_c,

    ROUND(
        MIN(w.temperature_c),
        2
    ) AS min_temperature_c,

    ROUND(
        MAX(w.temperature_c),
        2
    ) AS max_temperature_c,

    ROUND(
        SUM(w.precipitation_mm),
        2
    ) AS precipitation_mm,

    ROUND(
        MAX(w.wind_gusts_kmh),
        2
    ) AS max_wind_gust_kmh

FROM workspace.urbanpulse_gold.fact_weather w

INNER JOIN workspace.urbanpulse_gold.dim_date d
    ON w.observation_date_key = d.date_key

GROUP BY
    d.date_key,
    d.calendar_date

ORDER BY
    d.calendar_date DESC;

In [0]:
%sql
-- Inspecting transport and weather history together
SELECT
    d.calendar_date,

    COUNT(
        DISTINCT ls.line_status_key
    ) AS line_status_observations,

    SUM(
        CASE
            WHEN ls.is_disrupted THEN 1
            ELSE 0
        END
    ) AS disrupted_observations,

    ROUND(
        AVG(w.temperature_c),
        2
    ) AS avg_temperature_c,

    ROUND(
        AVG(w.precipitation_mm),
        2
    ) AS avg_precipitation_mm,

    ROUND(
        MAX(w.wind_gusts_kmh),
        2
    ) AS max_wind_gust_kmh

FROM workspace.urbanpulse_gold.dim_date d

LEFT JOIN workspace.urbanpulse_gold.fact_line_status ls
    ON d.date_key = ls.snapshot_date_key

LEFT JOIN workspace.urbanpulse_gold.fact_weather w
    ON d.date_key = w.observation_date_key

WHERE
    ls.line_status_key IS NOT NULL
    OR w.weather_observation_key IS NOT NULL

GROUP BY
    d.date_key,
    d.calendar_date

ORDER BY
    d.calendar_date DESC;

In [0]:
%sql
SELECT COUNT(*) AS facts
FROM workspace.urbanpulse_gold.fact_weather;